# 04 - Logistic Regression from Scratch

Now for the main part of the project - implementing logistic regression myself using just NumPy, no sklearn. Next notebook will compare this against sklearn's version.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)  # so results are reproducible

## Rebuilding the cleaned dataset

(same steps as notebook 02, just redoing it here so this notebook works on its own)

In [ ]:
df = pd.read_csv('../data/raw/student-mat.csv', sep=';')
df['pass'] = (df['G3'] >= 10).astype(int)

df_model = df.drop(columns=['G1', 'G2', 'G3'])

binary_cols = ['schoolsup', 'famsup', 'paid', 'activities', 'nursery',
               'higher', 'internet', 'romantic']
for col in binary_cols:
    df_model[col] = df_model[col].map({'yes': 1, 'no': 0})

df_model['school'] = df_model['school'].map({'GP': 1, 'MS': 0})
df_model['sex'] = df_model['sex'].map({'F': 1, 'M': 0})
df_model['address'] = df_model['address'].map({'U': 1, 'R': 0})
df_model['famsize'] = df_model['famsize'].map({'GT3': 1, 'LE3': 0})
df_model['Pstatus'] = df_model['Pstatus'].map({'T': 1, 'A': 0})

multi_cat_cols = ['Mjob', 'Fjob', 'reason', 'guardian']
df_model = pd.get_dummies(df_model, columns=multi_cat_cols, drop_first=True)

# get_dummies makes bool columns, cast everything to float for the math later
df_model = df_model.astype(float)

df_model.shape

In [ ]:
X = df_model.drop(columns=['pass']).values
y = df_model['pass'].values

X.shape, y.shape

## Train/test split

Doing this manually with a random permutation instead of sklearn's train_test_split, since the whole point of this notebook is "no sklearn". 80/20 split.

In [ ]:
def train_test_split_manual(X, y, test_size=0.2, seed=42):
    np.random.seed(seed)
    n = X.shape[0]
    indices = np.random.permutation(n)
    test_count = int(n * test_size)
    test_idx = indices[:test_count]
    train_idx = indices[test_count:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = train_test_split_manual(X, y)
X_train.shape, X_test.shape

## Feature scaling

Gradient descent works a lot better (and faster) when features are on a similar scale, so standardizing everything (mean 0, std 1). Important: fit the scaler on the TRAINING data only, then apply the same mean/std to the test set - otherwise it's a bit of data leakage from test into train.

In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1  # avoid dividing by zero for any constant columns

X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

## The logistic regression model itself

Sigmoid function, cost function (binary cross-entropy / log loss), and gradient descent to fit the weights. This is basically what I remember from the Mathematics for ML specialization + the stats course, just implementing the actual math this time instead of just reading about it.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
def compute_cost(X, y, weights, bias):
    n = X.shape[0]
    z = np.dot(X, weights) + bias
    predictions = sigmoid(z)

    # small epsilon so we never take log(0)
    eps = 1e-9
    cost = -(1/n) * np.sum(y * np.log(predictions + eps) + (1 - y) * np.log(1 - predictions + eps))
    return cost

In [ ]:
def gradient_descent(X, y, learning_rate=0.1, n_iterations=2000):
    n_samples, n_features = X.shape
    weights = np.zeros(n_features)
    bias = 0
    cost_history = []

    for i in range(n_iterations):
        z = np.dot(X, weights) + bias
        predictions = sigmoid(z)

        # gradients (derivative of the cost w.r.t weights and bias)
        dw = (1/n_samples) * np.dot(X.T, (predictions - y))
        db = (1/n_samples) * np.sum(predictions - y)

        weights -= learning_rate * dw
        bias -= learning_rate * db

        cost = compute_cost(X, y, weights, bias)
        cost_history.append(cost)

        # just to see progress without printing 2000 lines
        if i % 200 == 0:
            print(f'iteration {i}: cost = {cost:.4f}')

    return weights, bias, cost_history

I picked learning_rate=0.1 and 2000 iterations after just trying a couple of values - with 0.01 it was learning too slowly, with 1.0 the cost was jumping around instead of going down smoothly. Might tweak these more after seeing the cost plot below.

In [ ]:
weights, bias, cost_history = gradient_descent(X_train_scaled, y_train, learning_rate=0.1, n_iterations=2000)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(cost_history)
plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.title('Cost over training iterations')
plt.show()

If this curve is still dropping sharply at the end, it probably needs more iterations. If it flattens out early, we're fine (or could even use fewer iterations).

## Predictions and accuracy on the test set

In [ ]:
def predict(X, weights, bias, threshold=0.5):
    z = np.dot(X, weights) + bias
    probs = sigmoid(z)
    return (probs >= threshold).astype(int), probs

y_pred, y_probs = predict(X_test_scaled, weights, bias)

accuracy = np.mean(y_pred == y_test)
print(f'Test accuracy: {accuracy:.3f}')

## Next steps

- do the same thing with scikit-learn's LogisticRegression and compare accuracy + the actual learned coefficients
- proper evaluation metrics (precision, recall, F1, confusion matrix, ROC-AUC) for both